# Chicago Crime Forecasting & Tail-Risk Analysis

This notebook builds a grid-based, time-aware forecasting model for daily crime
counts across Chicago, then flags grid cells whose forecast errors show
heavy-tailed ("tail risk") behavior — i.e. locations where the model is
systematically surprised by extreme spikes.

**Pipeline overview**

1. Load and clean the raw Chicago crime dataset
2. Extract calendar/temporal features
3. Aggregate incidents onto a spatial grid and build a complete date x grid time series
4. Engineer lag, rolling-window, calendar, and holiday features
5. Train/validate an XGBoost regressor with a chronological (time-aware) split
6. Analyze residuals per grid cell using kurtosis to flag high tail-risk areas
7. Visualize trends, seasonality, model predictions, and risk classification

See the project [README](../README.md) for setup instructions, data sourcing,
and known limitations.


## 1. Environment Setup and Library Imports

Installs and imports everything needed for data loading, spatial processing,
modeling, statistical analysis, and visualization. Also sets global display
and plotting options, and fixes a random seed for reproducibility.

> If you're running locally with the packages from `requirements.txt` already
> installed, you can skip the `pip install` cell below — it's provided for
> convenience when running in Google Colab or a fresh environment.


In [ ]:
# Optional: uncomment if running in Colab or another environment where
# these packages are not yet installed. For local development, prefer
# `pip install -r requirements.txt` from the project root instead.

# !pip install -r ../requirements.txt


In [ ]:
import os
import gdown
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import MarkerCluster

# ML & utilities
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder

# Stats & XAI
from scipy.stats import kurtosis, skew
import shap

# Display helpers
pd.set_option('display.max_columns', 200)
plt.style.use('ggplot')

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Imports complete')


## 2. Data Loading and Initial Preprocessing

Loads the Chicago crime dataset and performs essential preprocessing:

- Converts the `Date` field to proper datetime format
- Filters the dataset to the desired analysis period (2020 onward)
- Removes entries with missing geographic coordinates

**Data source:** the full dataset is too large to check into this repository,
so it's fetched at runtime. By default this cell pulls a pre-exported CSV
from Google Drive via `gdown`. If you'd rather use your own copy (e.g.
downloaded directly from the
[Chicago Data Portal – Crimes dataset](https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-Present/ijzp-q8t2)),
place it at `data/crime_data.csv` and set `DOWNLOAD_DATA = False` below.


In [ ]:
DOWNLOAD_DATA = True
DATA_PATH = "../data/crime_data.csv"

# Google Drive file id for a pre-exported copy of the dataset.
# Replace with your own file id, or set DOWNLOAD_DATA = False and supply
# your own CSV at DATA_PATH.
GDRIVE_FILE_ID = "1MwcOtnfFqsyW30877VgtjqU5TJTPLo_J"

if DOWNLOAD_DATA:
    url = f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}"
    gdown.download(url, DATA_PATH, quiet=False)
elif not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"No file found at {DATA_PATH}. Either set DOWNLOAD_DATA = True, "
        "or place your own crime_data.csv at that path."
    )


In [ ]:
print('Reading CSV (this may take a while)...')
df = pd.read_csv(DATA_PATH, low_memory=False)
print('Raw rows:', len(df))

if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
else:
    raise ValueError('No Date column found in the dataset. Please verify column names.')

# Filter to recent years (2020 onward)
df = df[df['Date'].dt.year >= 2020].copy()
print('Filtered rows (2020+):', len(df))

# Keep only rows with lat/lon
lat_col = 'Latitude' if 'Latitude' in df.columns else 'Lat'
lon_col = 'Longitude' if 'Longitude' in df.columns else 'Long'
if lat_col not in df.columns or lon_col not in df.columns:
    raise ValueError('Latitude/Longitude columns not found. Columns present: ' + ','.join(df.columns))

df = df.dropna(subset=[lat_col, lon_col])
print('Rows after dropping missing coords:', len(df))


## 3. Temporal Feature Extraction

Extracts calendar-based features — year, month, day, hour, and day of week —
from the `Date` column. These help the model capture seasonal, daily, and
weekly crime patterns.


In [ ]:
# Basic temporal features
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['Hour'] = df['Date'].dt.hour
df['DayOfWeek'] = df['Date'].dt.dayofweek  # Monday=0

# Optionally compress categorical columns
cat_cols = [c for c in ['Primary Type', 'Description', 'Location Description'] if c in df.columns]
for c in cat_cols:
    df[c] = df[c].fillna('Unknown')

print('Sample rows:')
print(df.head())


## 4. Spatial Gridding and Time-Series Construction

Transforms point-based crime incidents into a structured grid-based time
series. Each incident is assigned to a grid cell (`Grid_ID`) based on
latitude/longitude divisions. A full *date x grid* scaffold is then built so
that days/locations with zero incidents are explicitly represented, giving a
complete dataset for time-series modeling.


In [ ]:
GRID_RESOLUTION = 0.005  # approx 500m

# Create grid coordinates
df['Grid_Lat'] = (df[lat_col] // GRID_RESOLUTION).astype(int)
df['Grid_Lon'] = (df[lon_col] // GRID_RESOLUTION).astype(int)

df['Grid_ID'] = df['Grid_Lat'].astype(str) + '_' + df['Grid_Lon'].astype(str)

print('Unique grid cells:', df['Grid_ID'].nunique())

# Create date-only column for daily aggregation
df['DateOnly'] = df['Date'].dt.floor('D')

# Aggregate counts per day per grid
df_ts = df.groupby(['DateOnly', 'Grid_ID']).size().reset_index(name='Crime_Count')

# Build full scaffold: full date range x all Grid_IDs (so missing days become 0)
all_dates = pd.date_range(df['DateOnly'].min(), df['DateOnly'].max(), freq='D')
all_grids = df['Grid_ID'].unique()

scaffold = pd.MultiIndex.from_product([all_dates, all_grids], names=['DateOnly', 'Grid_ID']).to_frame(index=False)

# Merge and fill missing counts with 0
scaffold = pd.merge(scaffold, df_ts, on=['DateOnly', 'Grid_ID'], how='left')
scaffold['Crime_Count'] = scaffold['Crime_Count'].fillna(0).astype(int)

print('Scaffold rows:', len(scaffold))


## 5. Feature Engineering (Lags, Rolling Means, Calendar, Holidays)

Builds the predictive feature set for the model:

- 1-day lagged crime counts
- 7-day rolling average of past counts
- Calendar features (month, day of week)
- US holiday indicator

> **Note on weather features:** `Avg_Temp` and `Precip_mm` are included in the
> feature list below as placeholders for future weather-data integration.
> They aren't populated by this notebook, so they're filled with `0` and
> currently contribute no signal to the model. See the README's "Known
> Limitations / Future Work" section for details.


In [ ]:
# Merge back grid lat/lon for mapping later
grid_coords = df.groupby('Grid_ID')[[lat_col, lon_col]].mean().reset_index()

# Work on scaffold: sort and create lag features per grid
scaffold = scaffold.sort_values(['Grid_ID', 'DateOnly']).reset_index(drop=True)

# Lag features: 1-day lag and 7-day rolling mean (computed on past values only)
scaffold['Crime_Lag_1'] = scaffold.groupby('Grid_ID')['Crime_Count'].shift(1).fillna(0)
scaffold['Crime_Roll7_mean'] = (
    scaffold.groupby('Grid_ID')['Crime_Count']
    .shift(1)
    .rolling(window=7, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
    .fillna(0)
)

# Calendar
scaffold['DayOfWeek'] = scaffold['DateOnly'].dt.dayofweek
scaffold['Month'] = scaffold['DateOnly'].dt.month

# Holiday flag
try:
    import holidays
    us_holidays = holidays.UnitedStates()
    scaffold['Is_Holiday'] = scaffold['DateOnly'].isin(us_holidays).astype(int)
except Exception:
    scaffold['Is_Holiday'] = 0

# Final feature list (extend this as needed)
features = ['Crime_Lag_1', 'Crime_Roll7_mean', 'DayOfWeek', 'Month', 'Avg_Temp', 'Precip_mm', 'Is_Holiday']

# Ensure any missing features are filled (see note above re: weather features)
for f in features:
    if f not in scaffold.columns:
        scaffold[f] = 0

print('Features prepared:', features)


## 6. Time-Aware Train/Validation Split

To respect chronological order in time-series forecasting, this splits the
dataset by date rather than randomly: the most recent 10% of dates are held
out as a validation set. This avoids data leakage and gives a realistic
estimate of forecasting performance.


In [ ]:
# Use last 10% of dates as validation (time-based split)
unique_dates = scaffold['DateOnly'].sort_values().unique()
split_idx = int(len(unique_dates) * 0.9)
train_dates = unique_dates[:split_idx]
val_dates = unique_dates[split_idx:]

train_df = scaffold[scaffold['DateOnly'].isin(train_dates)].copy()
val_df = scaffold[scaffold['DateOnly'].isin(val_dates)].copy()

# Prepare X and y
X_train = train_df[features].copy()
y_train = train_df['Crime_Count'].astype(float)
X_val = val_df[features].copy()
y_val = val_df['Crime_Count'].astype(float)

print('Train rows:', len(X_train), 'Val rows:', len(X_val))


## 7. XGBoost Model Training

Trains an XGBoost regression model to forecast crime intensity for each grid
cell and date, using the engineered temporal, spatial, and (placeholder)
weather features. Validation RMSE is printed to quantify prediction accuracy.


In [ ]:
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    tree_method='hist',
    random_state=RANDOM_STATE
)

xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)])

# Predict
y_pred_val = xgb_model.predict(X_val)
# Enforce non-negative predictions (crime counts can't be negative)
y_pred_val = np.maximum(0, y_pred_val)

rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
print(f'Validation RMSE: {rmse:.3f}')

# Attach predictions and residuals into val_df
val_df = val_df.reset_index(drop=True)
val_df['Predicted'] = y_pred_val
val_df['Residual'] = val_df['Crime_Count'] - val_df['Predicted']


## 8. Tail-Risk Analysis Using Residual Kurtosis

Evaluates how well the model performs across different grid cells by
analyzing the distribution of residuals (errors) per grid. High positive
kurtosis indicates a heavy-tailed error distribution — a signal that a
location has unpredictable or extreme crime spikes the model tends to miss.
Each grid cell is classified as either **High Tail Risk** or **Low Risk**.


In [ ]:
# Compute residual moments per grid
risk_df = val_df.groupby('Grid_ID')['Residual'].agg(
    Residual_Mean='mean',
    Residual_Variance='var',
    Residual_Kurtosis=lambda x: kurtosis(x, fisher=True, bias=False) if len(x) > 2 else 0
).reset_index()

# Set threshold
KURTOSIS_THRESHOLD = 2.5
risk_df['High_Tail_Risk'] = risk_df['Residual_Kurtosis'] > KURTOSIS_THRESHOLD

print('Number of high-tail-risk grids:', risk_df['High_Tail_Risk'].sum())
print(risk_df.sort_values('Residual_Kurtosis', ascending=False).head())


## 9. Identifying the Highest-Risk Grid Cell

Selects the single grid cell with the highest residual kurtosis among those
flagged as high tail-risk, for closer inspection.


In [ ]:
# Choose top grid by kurtosis
high_risk_grids = risk_df[risk_df['High_Tail_Risk']].sort_values('Residual_Kurtosis', ascending=False)
if len(high_risk_grids) == 0:
    print('No high tail risk grids found with threshold:', KURTOSIS_THRESHOLD)
else:
    target_grid = high_risk_grids.iloc[0]['Grid_ID']
    print('Target high-risk grid:', target_grid)


## 10. Visualizations

Summarizes the results: the overall daily crime trend, seasonality by day of
week and month, the highest-predicted-crime grid cells, the distribution of
residual kurtosis, the risk classification split, and how predicted crime
volume relates to tail risk.


In [ ]:
print("Visualization block loaded successfully.")

# Aggregate predicted values per grid to create pred_agg
pred_agg = val_df.groupby('Grid_ID')['Predicted'].agg(
    Predicted_Mean='mean',
    Predicted_Sum='sum'
).reset_index()

# 1) Time-Series Plot
plt.figure(figsize=(14, 5))
daily_series = scaffold.groupby('DateOnly')['Crime_Count'].sum()
plt.plot(daily_series.index, daily_series.values)
plt.title("Total Daily Crime in Chicago (2020+)")
plt.xlabel("Date")
plt.ylabel("Crime Count")
plt.grid(True)
plt.tight_layout()
plt.show()

# 2) Heatmap: DayOfWeek x Month
plt.figure(figsize=(10, 6))
heat_df = scaffold.groupby(['DayOfWeek', 'Month'])['Crime_Count'].mean().unstack().fillna(0)
sns.heatmap(heat_df, cmap="viridis", linewidths=0.5, annot=False)
plt.title("Heatmap of Avg Crime by DayOfWeek x Month")
plt.xlabel("Month")
plt.ylabel("DayOfWeek (0 = Monday)")
plt.tight_layout()
plt.show()

# 3) Top 20 High-Crime Grids
plt.figure(figsize=(12, 5))
top_20 = pred_agg.sort_values("Predicted_Mean", ascending=False).head(20)
plt.bar(top_20["Grid_ID"], top_20["Predicted_Mean"])
plt.xticks(rotation=90)
plt.title("Top 20 Highest Crime Grid Cells (Predicted Mean)")
plt.xlabel("Grid ID")
plt.ylabel("Predicted Mean Crime Count")
plt.tight_layout()
plt.show()

# 4) Kurtosis Distribution
plt.figure(figsize=(10, 5))
plt.hist(risk_df["Residual_Kurtosis"], bins=40, color="purple")
plt.title("Distribution of Residual Kurtosis Across Grid Cells")
plt.xlabel("Kurtosis")
plt.ylabel("Number of Grid Cells")
plt.tight_layout()
plt.show()

# 5) Risk Flag Bar Chart
plt.figure(figsize=(6, 4))
risk_counts = risk_df["High_Tail_Risk"].value_counts()
plt.bar(["Low Risk", "High Tail Risk"], risk_counts.values, color=["green", "red"])
plt.title("Count of Risk Categories (Kurtosis-Based)")
plt.ylabel("Number of Grid Cells")
plt.tight_layout()
plt.show()

# 6) Scatter: Predicted Mean vs Kurtosis
plt.figure(figsize=(8, 6))
merged = pd.merge(risk_df, pred_agg, on="Grid_ID", how="inner")
plt.scatter(
    merged["Predicted_Mean"],
    merged["Residual_Kurtosis"],
    c=merged["High_Tail_Risk"].map({True: "red", False: "blue"})
)
plt.xlabel("Predicted Mean Crime Count")
plt.ylabel("Residual Kurtosis")
plt.title("Predicted Crime vs Tail Risk")
plt.grid(True)
plt.show()

print("All visualizations generated successfully.")
